# Swept solid beam with imperfection — `replace_line` → `addPipe` → direct-PG BCs

Same "imperfection baked into the geometry" idea as the two-notebook
`buckleUP` series, now with a **3-D solid** body instead of a line
element:

1. Build a straight path with `add_line`
2. Retrofit it with a half-sine imperfection via `replace_line`
3. Build the cross-section profile as a planar face at `x = 0`
4. **Sweep** the profile along the imperfect path with
   `gmsh.model.occ.addPipe` — the resulting solid inherits the path
   curvature so the imperfection is in the volume itself
5. Delete the leftover 1-D path segments (they would mesh into
   stray line2 elements whose nodes have no tet4 stiffness)
6. Label the volume and the two end caps as **physical groups**
7. Generate a tet4 mesh
8. In OpenSees: query the end-face nodes directly through the PG
   labels and apply `fix` / `load` to them — no reference points,
   no `node_to_surface` couplings, no `rigidLink` chain

This is the simplest path from "I have a solid geometry" to "my FEM
has clean boundary conditions." Rather than using `node_to_surface`
to drive a single reference point that's rigidly linked to an entire
face, we let the **physical group do the bookkeeping**: the mesh
still carries face-node-set membership through `pin_face` and
`roller_face` labels, and OpenSees iterates those sets to apply the
per-node BCs. Applying a pure end moment becomes distributing an
equivalent force pattern across the face (linear in `z`), which we
do explicitly in code.

### What this workflow does vs. what a line beam does

* Captures the **full 3-D cross-section deformation** — flanges
  bending in plan under twist, local stress concentrations at the
  supports, warping at the ends.
* Loses the clean closed-form comparison to `M_cr = (π/L)·√(EI·GJ)`
  that the line beam gives exactly. The classical LTB formula
  corresponds to a warping-free line element; the solid naturally
  includes warping and so its critical moment is typically higher.
* Is a demonstration of **workflow**, not a buckling analysis —
  `FourNodeTetrahedron` is total-Lagrangian but the small-strain
  assumption + corotational coupling needed to capture the full LTB
  limit load is more than we're doing here. We run in linear-elastic
  pre-buckling regime and show the imperfect geometry plus the direct
  PG-based BC application.

### Why not `node_to_surface` for the ends?

We looked at it and backed off: the combination of
`rigidLink` + `equalDOF` + tet4 on a curved solid creates a
numerically fragile constraint chain (the ref-point rotational DOFs
have no direct stiffness contribution from the `ndf=3` solid face
nodes, so the reduced system tends to be ill-conditioned). PG-based
direct BC application is the pragmatic alternative: every face node
carries the support condition independently, the stiffness matrix is
just the tet4 stiffness with some rows zeroed out, and the solver is
happy.

In [ ]:
from apeGmsh import apeGmsh, Results
import gmsh
import numpy as np
import matplotlib.pyplot as plt

# ---- Steel material ----------------------------------------------
E   = 200_000.0
nu  = 0.3
G   = E / (2.0 * (1 + nu))

# ---- Rectangular section (tall narrow strip, LTB-prone) ---------
# Tall narrow rectangle meshes cleanly in tet4 — the thin web and
# flanges of a W-section produce sliver elements unless you bury
# the mesh under local refinement.
b_depth = 200.0      # strong-axis direction (z)
t_thk   = 20.0       # weak-axis direction (y)

A        = b_depth * t_thk
I_strong = t_thk * b_depth ** 3 / 12.0
I_weak   = b_depth * t_thk ** 3 / 12.0
# Approx St.-Venant torsion for a narrow rectangle:
J = (1.0 / 3.0) * b_depth * t_thk ** 3 * (1 - 0.63 * t_thk / b_depth)

# ---- Beam path + imperfection ------------------------------------
L       = 4000.0
delta_0 = L / 1000.0
lc      = 30.0        # global mesh size

# ---- Line-element reference LTB critical moment -----------------
M_cr_line = (np.pi / L) * np.sqrt(E * I_weak * G * J)

print(f'Rectangle {b_depth:.0f} x {t_thk:.0f} mm')
print(f'  A        = {A:.0f} mm^2')
print(f'  I_strong = {I_strong:.3e} mm^4')
print(f'  I_weak   = {I_weak:.3e} mm^4  (I_strong/I_weak = {I_strong/I_weak:.0f})')
print(f'  J        = {J:.0f} mm^4')
print()
print(f'L_b = {L:.0f} mm   delta_0 = L/1000 = {delta_0:.2f} mm')
print(f'Line-element M_cr (no warping) = {M_cr_line/1e6:.2f} kN*m')

## 1. Build the imperfect path, the profile, and the swept solid

Three Gmsh operations in one cell:

1. A straight path line from `(0, 0, 0)` to `(L, 0, 0)`, retrofitted
   with `replace_line(shape='sine', direction=(0, 1, 0))` to seed a
   half-sine imperfection perpendicular to the beam in global Y.
2. The rectangular cross-section as a 4-vertex planar face at `x = 0`.
3. `addWire` on the 16 sine segments + `addPipe` sweeps the profile
   along the imperfect path to produce a solid that inherits the
   curvature.

Two cleanup steps are critical:

* `remove([(2, prof_face)], recursive=True)` — removes the original
  profile surface (and its child lines/points). Without this, the
  original face persists at `x = 0` alongside the pipe's own start
  cap, and when we ask for `pin_face` by bounding box we'd pick up
  both, meshing them twice and creating duplicate-location nodes.
* `remove([(1, t) for t in path_tags], recursive=False)` — the
  imperfect path segments live along the centroid line of the solid
  (i.e. **inside** the volume). If left in the model, Gmsh meshes
  them as `line2` elements whose nodes sit interior to the tets and
  never appear in the tet connectivity. Those nodes are null-space
  modes in the stiffness matrix and make UMFPACK fail with
  "numeric analysis returns 1". Deleting the path lines after the
  pipe is built removes them cleanly.

In [ ]:
m = apeGmsh(model_name='swept_solid', verbose=False)
m.begin()

# (a) Imperfect path
p0 = m.model.geometry.add_point(0, 0, 0, lc=lc)
p1 = m.model.geometry.add_point(L, 0, 0, lc=lc)
path_line = m.model.geometry.add_line(p0, p1)
path_tags = m.model.geometry.replace_line(
    path_line,
    magnitude=delta_0,
    direction=(0, 1, 0),
    shape='sine',
    n_segments=16,
)
print(f'imperfect path: {len(path_tags)} sine segments')

# (b) Rectangular profile at x = 0, plane y-z
corners = [
    (0, -t_thk/2, -b_depth/2),
    (0,  t_thk/2, -b_depth/2),
    (0,  t_thk/2,  b_depth/2),
    (0, -t_thk/2,  b_depth/2),
]
cp = [gmsh.model.occ.addPoint(*c) for c in corners]
cl = [gmsh.model.occ.addLine(cp[i], cp[(i + 1) % 4]) for i in range(4)]
prof_loop = gmsh.model.occ.addCurveLoop(cl)
prof_face = gmsh.model.occ.addPlaneSurface([prof_loop])
gmsh.model.occ.synchronize()

# (c) Sweep
wire = gmsh.model.occ.addWire(path_tags, checkClosed=False)
gmsh.model.occ.addPipe([(2, prof_face)], wire)
gmsh.model.occ.synchronize()

# --- Cleanup ---
# Remove the original profile surface + its sub-entities (lines, points)
gmsh.model.occ.remove([(2, prof_face)], recursive=True)
# Remove the 1-D path segments (they sit inside the volume now)
gmsh.model.occ.remove([(1, t) for t in path_tags], recursive=False)
# And their endpoint 0-D points
for pp in (p0, p1):
    try:
        gmsh.model.occ.remove([(0, pp)], recursive=False)
    except Exception:
        pass
gmsh.model.occ.synchronize()

vols = gmsh.model.getEntities(3)
print(f'volumes after sweep + cleanup: {[v[1] for v in vols]}')
print(f'total dim-2 surfaces:          {len(gmsh.model.getEntities(2))}')
print(f'total dim-1 curves:            {len(gmsh.model.getEntities(1))}')

## 2. Label the volume and the two end caps as physical groups

Scan every dim-2 surface. Any surface whose bounding box has
`x_min == x_max` (to within a tolerance) is a cap. Group them by
which end they sit on, promote the volume + the two cap sets to
physical groups with clean names:

* `pg_beam` — the swept solid volume
* `pin_face` — the cap at `x = 0`
* `roller_face` — the cap at `x = L`

These physical groups are all apeGmsh needs to route labels into
`fem.nodes.get_ids(target=...)` later. No reference points, no
constraint definitions — the PGs **are** the BC/load handles.

In [ ]:
TOL = 0.1
start_caps, end_caps = [], []
for (d, t) in gmsh.model.getEntities(2):
    bb = gmsh.model.getBoundingBox(d, t)
    if abs(bb[3] - bb[0]) < TOL:   # x_min == x_max
        if abs(bb[0]) < TOL:
            start_caps.append(t)
        elif abs(bb[0] - L) < TOL:
            end_caps.append(t)
print(f'start caps (x=0): {start_caps}')
print(f'end caps (x=L):   {end_caps}')

vol_tags = [v[1] for v in gmsh.model.getEntities(3)]
gmsh.model.addPhysicalGroup(3, vol_tags,   tag=-1, name='pg_beam')
gmsh.model.addPhysicalGroup(2, start_caps, tag=-1, name='pin_face')
gmsh.model.addPhysicalGroup(2, end_caps,   tag=-1, name='roller_face')
gmsh.model.occ.synchronize()

## 3. Mesh the solid

Global element size `lc = 30 mm` gives ~10 elements across the
200 mm strong-axis depth and ~2 through the 20 mm thickness — coarse
but enough for a workflow demo. A W-section with 12 mm flanges
would need a smaller `lc` or local refinement to avoid sliver tets.

In [ ]:
m.mesh.sizing.set_global_size(lc)
m.mesh.generation.generate(dim=3)

fem = m.mesh.queries.get_fem_data(remove_orphans=True)
m.end()

print(f'total nodes in mesh: {len(fem.nodes.ids)}')
for g in fem.elements:
    print(f'  {g.type_name:6s} n={len(g)}')
print()
# PG-based filtering: only solid volume nodes go to OpenSees. Nodes
# that Gmsh created from 1-D/2-D entities on the boundary are in the
# mesh but are automatically shared with the tet4 elements (conformal
# mesh), so they come through pg_beam just fine.
print(f'pg_beam  nodes: {len(fem.nodes.get_ids(target="pg_beam"))}')
print(f'pin_face nodes: {len(fem.nodes.get_ids(target="pin_face"))}')
print(f'roller_face nodes: {len(fem.nodes.get_ids(target="roller_face"))}')

## 4. Emit the OpenSees model

Everything goes through PG-based queries:

* **`fem.nodes.get(target='pg_beam')`** — only the solid volume
  nodes are emitted. Any stray line-mesh nodes (from surface
  boundaries, leftover path curves, etc.) are filtered out at source,
  so they never end up as floating DOFs in OpenSees.
* **`FourNodeTetrahedron`** elements for every `tet4` group.
* **BCs applied directly on `pin_face` / `roller_face`** nodes —
  no ref point, no rigid link, no equalDOF.

The fork-support pattern for LTB is simulated this way: at every
face node we fix `uy` and `uz` (locks lateral motion + twist because
the face is a rigid plane), and leave `ux` free (allows rotation of
the face about y and z via differential ux among face nodes). One
ux DOF gets locked at the pin-face centroid to remove rigid-body
translation in x. The resulting constraint set is:

| end         | per face node | extra constraint                    |
|-------------|---------------|-------------------------------------|
| `pin_face`  | `uy = uz = 0` | `ux = 0` at the centre-most node    |
| `roller_face`| `uy = uz = 0`| *(none — axial free)*               |

In [ ]:
import openseespy.opensees as ops

ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 3)
ops.timeSeries('Linear', 1)

# --- Solid tet nodes (filtered to pg_beam — no orphans) ----------
n_solid = 0
for nid, xyz in fem.nodes.get(target='pg_beam'):
    ops.node(int(nid), *xyz)
    n_solid += 1
print(f'solid nodes emitted: {n_solid}')

# --- Material + tet4 elements ------------------------------------
ops.nDMaterial('ElasticIsotropic', 1, E, nu)
n_tet = 0
for group in fem.elements.get(element_type='tet4'):
    for eid, conn in group:
        ops.element('FourNodeTetrahedron', int(eid), *conn, 1)
        n_tet += 1
print(f'tet4 elements: {n_tet}')

# --- BCs ---------------------------------------------------------
# We find the pin-face centroid node first so we can give it the
# full (1, 1, 1) fix and exclude it from the per-face uy/uz loop
# below (OpenSees refuses to add two SP constraints on the same DOF
# of the same node).
pin_ids    = np.array(
    [int(n) for n in fem.nodes.get_ids(target='pin_face')])
roller_ids = np.array(
    [int(n) for n in fem.nodes.get_ids(target='roller_face')])
pin_coords    = fem.nodes.get_coords(target='pin_face')
roller_coords = fem.nodes.get_coords(target='roller_face')

d_centroid   = np.linalg.norm(pin_coords, axis=1)
centroid_idx = int(np.argmin(d_centroid))
centroid_nid = int(pin_ids[centroid_idx])

# Full (ux, uy, uz) anchor at the centroid — removes rigid body
# translation in x.
ops.fix(centroid_nid, 1, 1, 1)

# Pin face, excluding the centroid (already handled above).
for i, nid in enumerate(pin_ids):
    if i == centroid_idx:
        continue
    ops.fix(int(nid), 0, 1, 1)    # uy, uz locked; ux free

# Roller face — uy, uz locked, ux free.
for nid in roller_ids:
    ops.fix(int(nid), 0, 1, 1)

print(f'\nBCs applied:')
print(f'  centroid anchor node {centroid_nid}: ux=uy=uz=0')
print(f'  pin_face    — {len(pin_ids) - 1} nodes with uy=uz=0')
print(f'  roller_face — {len(roller_ids)} nodes with uy=uz=0')

## 5. Apply end moments as distributed force couples and sweep

Since we do not have a reference point that we can load with a pure
`My`, we **distribute** the moment across the end face as a linear
force pattern in the axial direction:

$$
F_{x,i} \;=\; \frac{M_y}{\sum_j z_j^{2}} \; z_i
$$

At the **pin face** we apply `+F_{x,i}` and at the **roller face**
the opposite sign, so the internal moment is uniform `M_y` along the
span. The net horizontal force on each face is zero because
`Σ z_i = 0` for a section centred at the origin — no rigid-body
translation driving, just a pure force couple.

Linear elastic analysis, one load step, sweep to ~60 % of the
line-element `M_cr` so we stay comfortably in pre-buckling.

In [ ]:
# --- Build the force-couple pattern for a unit reference moment --
# F_x,i = M * z_i / sum(z_j^2). We use M_ref = 1 N*mm, so each
# face-node force is F_x,i = z_i / sum(z^2). The load factor in the
# integrator then multiplies the whole pattern, giving an effective
# applied moment = load_factor N*mm.
sum_z2_pin  = float(np.sum(pin_coords[:, 2] ** 2))
sum_z2_roll = float(np.sum(roller_coords[:, 2] ** 2))

ops.pattern('Plain', 1, 1)
for nid, c in zip(pin_ids, pin_coords):
    fx = +c[2] / sum_z2_pin
    ops.load(int(nid), fx, 0.0, 0.0)
for nid, c in zip(roller_ids, roller_coords):
    fx = -c[2] / sum_z2_roll
    ops.load(int(nid), fx, 0.0, 0.0)

# --- Solver ------------------------------------------------------
ops.constraints('Plain')
ops.numberer('RCM')
ops.system('UmfPack')
ops.test('NormDispIncr', 1.0e-7, 100)
ops.algorithm('Linear')

M_target = 0.6 * M_cr_line       # stay in linear pre-buckling
n_steps  = 20
dM       = M_target / n_steps
ops.integrator('LoadControl', dM)
ops.analysis('Static')

# --- Midspan probe -----------------------------------------------
beam_ids    = np.array(
    [int(n) for n in fem.nodes.get_ids(target='pg_beam')])
beam_coords = fem.nodes.get_coords(target='pg_beam')
d_mid = np.linalg.norm(
    beam_coords - np.array([L/2, 0, 0]), axis=1)
mid_nid = int(beam_ids[int(np.argmin(d_mid))])
print(f'midspan probe node {mid_nid} at '
      f'{tuple(beam_coords[int(np.argmin(d_mid))].round(1))}')

# Build an index: for each OpenSees-emitted node, the row in the
# full fem.nodes.ids array. Used to populate the (N, 3) field that
# Results.from_fem expects while only querying nodeDisp on the
# subset we actually emitted.
emitted_ids = set(int(n) for n in beam_ids)
id_to_row   = {int(nid): i for i, nid in enumerate(fem.nodes.ids)}

hist_M    = []
hist_midy = []
hist_midz = []
disp_per_step: list[np.ndarray] = []
n_nodes = len(fem.nodes.ids)

for step in range(n_steps):
    ok = ops.analyze(1)
    if ok != 0:
        print(f'failed at step {step + 1}, ok={ok}')
        break
    M_now = (step + 1) * dM
    d = ops.nodeDisp(mid_nid)
    hist_M.append(M_now)
    hist_midy.append(d[1])
    hist_midz.append(d[2])

    # Capture displacement only on emitted (pg_beam) nodes. Stray
    # mesh nodes from dim-1/dim-2 entities that were not emitted to
    # OpenSees stay at zero — they don't participate in the viewer's
    # deformed shape.
    d_full = np.zeros((n_nodes, 3), dtype=np.float64)
    for nid in emitted_ids:
        row = id_to_row[nid]
        di  = ops.nodeDisp(nid)
        d_full[row, 0] = di[0]
        d_full[row, 1] = di[1]
        d_full[row, 2] = di[2]
    disp_per_step.append(d_full)

print(f'converged in {len(hist_M)} of {n_steps} steps')
print(f'final M       = {hist_M[-1]/1e6:.3f} kN*m  '
      f'({hist_M[-1]/M_cr_line*100:.1f}% M_cr_line)')
print(f'final mid uy  = {hist_midy[-1]:+.5f} mm')
print(f'final mid uz  = {hist_midz[-1]:+.5f} mm')

## 6. M vs midspan deflections

Linear elastic pre-buckling regime — both `uz` (strong-axis vertical
deflection) and `uy` (weak-axis lateral deflection) grow linearly
with `M`, with `uy` much smaller than `uz` because most of the load
goes into strong-axis bending. The small-but-nonzero `uy` is driven
by the initial imperfection seeded into the geometry by
`replace_line`.

In [ ]:
hist_M    = np.asarray(hist_M)
hist_midy = np.asarray(hist_midy)
hist_midz = np.asarray(hist_midz)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
ax.plot(hist_midz, hist_M / 1e6, 'o-', lw=1.4, ms=3, color='tab:blue')
ax.set_xlabel('midspan uz [mm]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Strong-axis bending (vertical)')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(hist_midy, hist_M / 1e6, 'o-', lw=1.4, ms=3, color='tab:red')
ax.axvline(delta_0, color='tab:gray', lw=0.8, ls=':',
           label=f'\u03b4\u2080 = L/1000 = {delta_0:.2f} mm')
ax.set_xlabel('midspan uy [mm]')
ax.set_ylabel('M  [kN*m]')
ax.set_title('Weak-axis deflection (imperfection driven)')
ax.grid(True, alpha=0.3)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

## 7. Time-series viewer

Push the displacement field from every load step into
`Results.from_fem(steps=...)` with `time = M` so you can scrub through
the loading in the viewer and watch the imperfect solid progressively
bend under the distributed moment pattern.

In [ ]:
steps = []
for M_i, d_step in zip(hist_M, disp_per_step):
    u_mag = np.linalg.norm(d_step, axis=1)
    steps.append({
        'time': float(M_i),
        'point_data': {
            'Displacement': d_step,
            '|U|':          u_mag,
            'Uy':           d_step[:, 1],
            'Uz':           d_step[:, 2],
        },
    })

print(f'time-series steps: {len(steps)}')
print(f'time range       : {steps[0]["time"]/1e6:.2f} to '
      f'{steps[-1]["time"]/1e6:.2f} kN*m')

results = Results.from_fem(
    fem,
    steps=steps,
    name='LTB_swept_solid',
)
results.viewer(blocking=False)